# 41 · Prompt caching
Level L4 · Part 5 · professional · ~35 min · builds on 12, 31

## Goal
Structure Scout's requests so the long, stable start (tool list, system prompt, notes) can be cached:
place breakpoints, respect the TTL and the minimum size, measure the **cache hit rate** over 5 turns,
and debug a 0% hit rate caused by a timestamp in the system prompt.

## The idea
Lesson 12 showed Scout pays for its whole prompt on every call. Lesson 31 said providers can reuse a
request's **stable prefix**. **Prompt caching** is that reuse: the provider keeps the work it did on
the start of your prompt for a few minutes, and the next request that starts with exactly the same
tokens pays about a tenth of the input price for that part. The match is exact and from the first
token: change one character near the top and nothing after it can be reused. So the rule is simple:
what never changes goes first, what changes goes last, and you check the hit rate instead of hoping.

Analogy: a shop that keeps your half-built order on the counter for 5 minutes. Come back with the same
order and they only add the new items; change the first line and they start again.

## Picture
```mermaid
flowchart LR
    T["tools"] --> S["system prompt + notes"] --> H["earlier turns"] --> Q["new question"]
    S -. "breakpoint 1" .-> C[("provider cache<br/>5 min TTL")]
    H -. "breakpoint 2 (moves)" .-> C
    C -->|"same prefix: read at 0.1x"| B["bill"]
    Q -->|"new tokens: full price"| B
```

## Step by step
### 1. Start the recorder
Gemini calls are recorded once to `../data/llm_cassettes/41_prompt-caching.yaml` and replayed after
that, with no key (lesson 07).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("41_prompt-caching")

LLM mode: replay (41_prompt-caching.yaml)


### 2. Setup from lessons 12 and 31
The client and pinned model, lesson 31's token counter (tiktoken `o200k_base` from the local cache, an
estimate) and its three long rover notebooks, generated with the same seed.

In [2]:
import hashlib, json, os, random
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
os.environ["TIKTOKEN_CACHE_DIR"] = "../data/tiktoken_cache"
import tiktoken
ENC = tiktoken.get_encoding("o200k_base")

def tokens(text: str) -> int:
    return len(ENC.encode(text))

rng = random.Random(31)
def rover_log(first_sol: int, n: int) -> str:
    return "\n".join(f"Sol {s}: drove {rng.randint(0, 120)} m, wind {rng.randint(1, 15)} m/s, dust opacity "
                     f"{rng.uniform(0.2, 1.5):.2f}, battery {rng.randint(40, 100)}%, photos {rng.randint(0, 60)}."
                     for s in range(first_sol, first_sol + n))

The notebooks, lesson 31's tool schemas, and a long system prompt. This time Scout puts all three
notebooks in the prompt: lesson 31's "everything up front" design. It is the case where caching pays
most, because about 8k tokens (tiktoken estimate) repeat on every call.

In [3]:
NOTEBOOKS = {1: rover_log(1, 100), 2: rover_log(101, 100),
             3: rover_log(201, 55) + "\nReminder for the talk: a day on Mars (a sol) lasts 24 hours and 39 minutes."}

def schema(name: str, description: str, prop: str, kind: str) -> dict:
    return {"name": name, "description": description,
            "parameters": {"type": "object", "required": [prop], "properties": {prop: {"type": kind}}}}

SCHEMAS = [schema("search_notes", "Search the notebooks; returns id, title and size.", "query", "string"),
           schema("read_note", "Return the full text of one notebook by id.", "note_id", "integer"),
           schema("calculator", "Evaluate arithmetic with + - * / and brackets.", "expression", "string")]
RULES = ("You are Scout, a research assistant. Answer only from the field notebooks below. "
         "Answer in one short sentence and quote the sol number.")
SYSTEM_LONG = RULES + "\n\n" + "\n\n".join(f"# Notebook {i}\n{t}" for i, t in NOTEBOOKS.items())
print("tools:", tokens(json.dumps(SCHEMAS)), "tokens | system + notes:", tokens(SYSTEM_LONG), "tokens")

tools: 151 tokens | system + notes: 7714 tokens


### 3. Why it must be a prefix: the KV cache, from outside
For every prompt token the model computes some internal state (the **KV cache**) that depends on that
token *and every token before it*. A provider can store those states and reuse them only if everything
before is identical. A toy version: each "state" is a hash of the previous state plus the token.

In [4]:
def states(words: list[str]) -> list[str]:
    s, out = "", []
    for w in words:
        s = hashlib.sha256((s + w).encode()).hexdigest()[:6]
        out.append(s)
    return out

def reusable(a: list[str], b: list[str]) -> int:
    n = 0
    for x, y in zip(states(a), states(b)):
        if x != y:
            break
        n += 1
    return n

base = "You are Scout . Notes : sol 1 drove 40 m . sol 2 drove 12 m".split()
print("same prompt, new question at the end:", reusable(base + ["Q1"], base + ["Q2"]), "of", len(base) + 1)
changed = base.copy(); changed[2] = "Scout!"       # one word changed near the top
print("one word changed at position 2      :", reusable(base + ["Q1"], changed + ["Q2"]), "of", len(base) + 1)

same prompt, new question at the end: 17 of 18
one word changed at position 2      : 2 of 18


Same start: 17 of 18 states reused. One early change: only 2. That is why caching works on prefixes
only, and why the stable part must come first (lesson 31, step 8).

### 4. A provider cache you can watch
Gemini caches implicitly; Anthropic lets you mark **cache breakpoints**. To see the rules at work, we
build a small simulator that follows Anthropic's documented rules (prompt caching page, fetched
2026-10-09): prefix order tools, system, messages; at most 4 breakpoints; a minimum size; a 5-minute
TTL refreshed on every hit. First the rules and prices for `claude-sonnet-5-5`.

In [5]:
CACHE_RULES = {            # source: platform.claude.com/docs/en/build-with-claude/prompt-caching, 2026-10-09
    "model": "claude-sonnet-5-5", "input": 2.00, "output": 10.00,     # USD per million tokens
    "write": {"5m": 1.25, "1h": 2.0},    # cache writes cost more than normal input
    "read": 0.05,                        # 0.1x on most models; 0.05x on Sonnet 5.5 and Opus 5.5
    "min_tokens": 512, "max_breakpoints": 4, "lookback": 20,
}
TTL_SECONDS = {"5m": 300, "1h": 3600}

def prefix_key(blocks: list[dict]) -> str:
    return hashlib.sha256(json.dumps([b["text"] for b in blocks]).encode()).hexdigest()

print("read price:", CACHE_RULES["read"] * CACHE_RULES["input"], "$/MTok vs", CACHE_RULES["input"], "uncached")

read price: 0.1 $/MTok vs 2.0 uncached


The simulator. A request is a list of blocks; a block with `"cache": "5m"` is a breakpoint. On each call
it looks back from the last breakpoint for the longest prefix it has stored (a **hit**), then stores
the prefix at every breakpoint that is long enough (a **write**). Everything else is plain input.

In [6]:
class PrefixCache:
    def __init__(self, rules: dict = CACHE_RULES):
        self.rules, self.store = rules, {}                      # prefix key -> expiry time (s)

    def call(self, blocks: list[dict], now: float) -> dict:
        bps = [i for i, b in enumerate(blocks) if b.get("cache")]
        if len(bps) > self.rules["max_breakpoints"]:
            raise ValueError(f"{len(bps)} cache breakpoints; the API allows {self.rules['max_breakpoints']}")
        sizes = [tokens(b["text"]) for b in blocks]
        read, last = 0, (bps[-1] if bps else -1)
        for i in range(last, max(last - self.rules["lookback"], -1), -1):      # longest stored prefix
            if self.store.get(prefix_key(blocks[:i + 1]), -1) >= now:
                read = sum(sizes[:i + 1])
                break
        written = 0
        for i in bps:
            n, ttl = sum(sizes[:i + 1]), blocks[i]["cache"]
            if n >= self.rules["min_tokens"]:                   # too short: silently not cached
                self.store[prefix_key(blocks[:i + 1])] = now + TTL_SECONDS[ttl]   # write or refresh
                written = max(written, n - read)
        return {"input": sum(sizes) - read - written, "cache_write": written, "cache_read": read}

Scout's request, in the documented order. One breakpoint after the system prompt (tools + system +
notes), and one on the newest message, which moves forward every turn so the history is cached too.
Anthropic's "automatic caching" (one top-level `cache_control`) places that moving one for you.

In [7]:
def build_request(messages: list[dict], system: str = SYSTEM_LONG, ttl: str = "5m") -> list[dict]:
    blocks = [{"text": json.dumps(SCHEMAS)}, {"text": system, "cache": ttl}]           # breakpoint 1
    blocks += [{"text": f"{m['role']}: {m['text']}"} for m in messages]
    blocks[-1] = {**blocks[-1], "cache": ttl}                                           # breakpoint 2
    return blocks

QUESTIONS = ["How far did the rover drive on sol 12?", "What was the battery level on sol 150?",
             "Which sol had the most photos in notebook 3?", "How long is a day on Mars?",
             "What was the dust opacity on sol 77?"]
print([len(build_request([{"role": "user", "text": q}])) for q in QUESTIONS[:1]], "blocks on turn 1")

[3] blocks on turn 1


A 5-turn chat on the simulator, 30 seconds apart. The replies are canned: only the bill matters here.
`cost_cached` is lesson 12's `cost_usd` with the two extra cache prices.

In [8]:
def cost_cached(u: dict, out_tokens: int, rules: dict = CACHE_RULES, ttl: str = "5m") -> float:
    return (u["input"] * rules["input"] + u["cache_write"] * rules["input"] * rules["write"][ttl]
            + u["cache_read"] * rules["input"] * rules["read"] + out_tokens * rules["output"]) / 1_000_000

def sim_chat(cache: PrefixCache, make_user_text=lambda q, t: q, system_at=lambda t: SYSTEM_LONG,
             gap_s: float = 30, ttl: str = "5m") -> list[dict]:
    messages, rows = [], []
    for turn, q in enumerate(QUESTIONS):
        now = turn * gap_s
        messages.append({"role": "user", "text": make_user_text(q, now)})
        u = cache.call(build_request(messages, system_at(now), ttl), now)
        rows.append({**u, "cost": cost_cached(u, 20, ttl=ttl)})
        messages.append({"role": "assistant", "text": f"(answer {turn + 1}, about twenty tokens long)"})
    return rows

Run it and print each turn's bill.

In [9]:
def show(rows: list[dict]) -> None:
    for i, r in enumerate(rows, 1):
        print(f"turn {i}: {r['input']:>5} plain | {r['cache_write']:>5} written | {r['cache_read']:>5} read"
              f" | ${r['cost']:.5f}")
    print(f"total ${sum(r['cost'] for r in rows):.4f}")

good = sim_chat(PrefixCache())
show(good)

turn 1:     0 plain |  7878 written |     0 read | $0.01989
turn 2:     0 plain |    24 written |  7878 read | $0.00105
turn 3:     0 plain |    25 written |  7902 read | $0.00105
turn 4:     0 plain |    22 written |  7927 read | $0.00105
turn 5:     0 plain |    24 written |  7949 read | $0.00105
total $0.0241


Turn 1 writes about 8k tokens at 1.25x, so it costs a little *more* than no caching. Turns 2 to 5 read
them at 0.05x and write only the newest two messages. Nothing is "plain": the moving breakpoint sits
on the last block, so every token is either read or written.

### 5. Measure the hit rate
The **cache hit rate** is the share of input tokens served from the cache. Measure it per request,
never assume it. Compare the bill with what the same 5 turns cost with no caching at all.

In [10]:
def hit_rate(rows: list[dict]) -> float:
    total = sum(r["input"] + r["cache_write"] + r["cache_read"] for r in rows)
    return sum(r["cache_read"] for r in rows) / total

def no_cache_cost(rows: list[dict]) -> float:
    return sum(cost_cached({"input": r["input"] + r["cache_write"] + r["cache_read"],
                            "cache_write": 0, "cache_read": 0}, 20) for r in rows)

print(f"hit rate {hit_rate(good):.0%} | cached ${sum(r['cost'] for r in good):.4f}"
      f" vs uncached ${no_cache_cost(good):.4f}")

hit rate 80% | cached $0.0241 vs uncached $0.0803


About 80% of input tokens came from the cache and the bill fell by roughly three quarters. The first
turn can never hit, so longer chats get closer to the ideal.

### 6. The bug: a timestamp in the system prompt
A teammate adds the current time to the top of the system prompt, "so Scout knows the date". The
answers stay fine. The bill goes up. Same chat, same simulator:

In [11]:
def clock(t: float) -> str:
    s = 10 * 3600 + 41 * 60 + int(t)                      # the chat starts at 10:41:00
    return f"{s // 3600:02d}:{s // 60 % 60:02d}:{s % 60:02d}"

buggy = sim_chat(PrefixCache(), system_at=lambda t: f"Current time: 2026-10-09 {clock(t)}.\n" + SYSTEM_LONG)
show(buggy)
print(f"hit rate {hit_rate(buggy):.0%}")

turn 1:     0 plain |  7895 written |     0 read | $0.01994
turn 2:     0 plain |  7919 written |     0 read | $0.02000
turn 3:     0 plain |  7944 written |     0 read | $0.02006
turn 4:     0 plain |  7966 written |     0 read | $0.02012
turn 5:     0 plain |  7990 written |     0 read | $0.02017
total $0.1003
hit rate 0%


0%. Every turn writes the whole prefix again at 1.25x, so it costs more than no caching at all, and
nothing errors. How do you find it? Compare two consecutive requests and find where they first differ.

In [12]:
def first_difference(a: list[dict], b: list[dict]) -> str:
    ta, tb = "".join(x["text"] for x in a), "".join(x["text"] for x in b)
    n = next((i for i, (x, y) in enumerate(zip(ta, tb)) if x != y), min(len(ta), len(tb)))
    return f"identical for {tokens(ta[:n])} tokens, then {ta[n:n + 25]!r} vs {tb[n:n + 25]!r}"

msgs = [{"role": "user", "text": QUESTIONS[0]}]
stamped = lambda t: f"Current time: 2026-10-09 {clock(t)}.\n" + SYSTEM_LONG
print("buggy:", first_difference(build_request(msgs, stamped(0)), build_request(msgs, stamped(30))))

buggy: identical for 166 tokens, then '00.\nYou are Scout, a rese' vs '30.\nYou are Scout, a rese'


The requests split after a few dozen tokens, inside the clock. The fix: keep the system prompt
constant and put anything that changes in the newest user message, at the end.

In [13]:
fixed = sim_chat(PrefixCache(), make_user_text=lambda q, t: f"[sent 2026-10-09 {clock(t)}] {q}")
show(fixed)
print(f"hit rate {hit_rate(fixed):.0%}")

turn 1:     0 plain |  7894 written |     0 read | $0.01994
turn 2:     0 plain |    40 written |  7894 read | $0.00109
turn 3:     0 plain |    41 written |  7934 read | $0.00110
turn 4:     0 plain |    38 written |  7975 read | $0.00109
turn 5:     0 plain |    40 written |  8013 read | $0.00110
total $0.0243
hit rate 80%


Back to about 80%, and Scout still knows the time. Old user messages keep their own timestamps, which
never change once sent, so the history stays cacheable.

### 7. TTL and the minimum size
Two quiet ways to get misses. If the user pauses longer than the TTL, the entry expires. And a prefix
shorter than the minimum is simply not cached, with no error. Lesson 12's short prompt is an example.

In [14]:
slow = sim_chat(PrefixCache(), gap_s=400)                        # 6 min 40 s between turns
short_prompt = "You are Scout, a research assistant. Answer in one sentence from the user's notes."
small = sim_chat(PrefixCache(), system_at=lambda t: short_prompt)
print(f"400 s gaps, 5m TTL : hit rate {hit_rate(slow):.0%}")
print(f"short prompt       : {tokens(json.dumps(SCHEMAS)) + tokens(short_prompt)} tokens before breakpoint 1,"
      f" hit rate {hit_rate(small):.0%}")

400 s gaps, 5m TTL : hit rate 0%
short prompt       : 168 tokens before breakpoint 1, hit rate 0%


Slow turns get no hits. The short prompt plus 5 short turns never reaches 512 tokens, so nothing is
cached at all, and no error says so (here that costs little, because the prompt is small). For slow users a 1-hour TTL keeps the entry, at 2x for the write:

In [15]:
slow_1h = sim_chat(PrefixCache(), gap_s=400, ttl="1h")
print(f"400 s gaps, 1h TTL : hit rate {hit_rate(slow_1h):.0%}, ${sum(r['cost'] for r in slow_1h):.4f}"
      f" vs ${no_cache_cost(slow_1h):.4f} uncached")

400 s gaps, 1h TTL : hit rate 80%, $0.0361 vs $0.0803 uncached


### 8. The real thing: Scout on Gemini
Gemini 2.5 and newer cache **implicitly**: no breakpoints, the discount applies when a request starts
like a recent one. Vertex documents a 90% discount on cached tokens, a minimum of 4,096 tokens for the
Gemini 3 family, and the field `cached_content_token_count` in `usage_metadata`. Hits are best effort,
so we read the field. First a price table with the cached rate (Gemini pricing page, 2026-10-09).

In [16]:
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "cached": 0.03, "output": 2.50}}

def gemini_cost(prompt: int, cached: int, out: int, model: str = MODEL) -> float:
    p = PRICES_PER_MTOK[model]
    return ((prompt - cached) * p["input"] + cached * p["cached"] + out * p["output"]) / 1_000_000

DECLS = [types.FunctionDeclaration(name=s["name"], description=s["description"],
                                   parameters_json_schema=s["parameters"]) for s in SCHEMAS]
print("ready:", [d.name for d in DECLS])

ready: ['search_notes', 'read_note', 'calculator']


Scout's real chat as a function. The tools stay in the prompt (tool mode `NONE`: listed, not callable,
so Scout answers from the notes). `system_at` and `user_text` decide where the timestamp goes. Each
turn keeps its prompt, cached and output tokens, plus the request for debugging.

In [17]:
def real_chat(system_at=lambda t: SYSTEM_LONG, user_text=lambda q, t: q) -> tuple[list[dict], list]:
    rows, contents, requests = [], [], []
    for turn, q in enumerate(QUESTIONS):
        now = turn * 30
        cfg = types.GenerateContentConfig(system_instruction=system_at(now), max_output_tokens=60,
            tools=[types.Tool(function_declarations=DECLS)], automatic_function_calling=NO_AFC,
            tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="NONE")))
        contents.append(types.Content(role="user", parts=[types.Part(text=user_text(q, now))]))
        requests.append((system_at(now), list(contents)))
        resp = client.models.generate_content(model=MODEL, contents=contents, config=cfg)
        u = resp.usage_metadata
        rows.append({"prompt": u.prompt_token_count, "cached": u.cached_content_token_count or 0,
                     "out": (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)})
        contents.append(resp.candidates[0].content)
        print(f"turn {turn + 1}: {rows[-1]} | {resp.text.strip()[:50]}")
    return rows, requests

The fixed design first: constant system prompt, timestamp at the end of the user message.

In [18]:
real_fixed, fixed_requests = real_chat(user_text=lambda q, t: f"[sent 2026-10-09 {clock(t)}] {q}")

turn 1: {'prompt': 9779, 'cached': 8160, 'out': 13} | The rover drove 11 m on Sol 12.
turn 2: {'prompt': 9827, 'cached': 8152, 'out': 15} | The battery level was 42% on Sol 150.


turn 3: {'prompt': 9876, 'cached': 8144, 'out': 32} | Sol 213, Sol 235, and Sol 251 had the most photos 
turn 4: {'prompt': 9939, 'cached': 8136, 'out': 19} | A day on Mars (a sol) lasts 24 hours and 39 minute
turn 5: {'prompt': 9992, 'cached': 8128, 'out': 15} | The dust opacity was 1.33 on Sol 77.


Then the teammate's version: the timestamp at the top of the system prompt. Then we save both runs'
usage for the fallback.

In [19]:
real_buggy, buggy_requests = real_chat(system_at=lambda t: f"Current time: 2026-10-09 {clock(t)}.\n" + SYSTEM_LONG)
json.dump({"fixed": real_fixed, "buggy": real_buggy}, open("../data/recorded/41.json", "w"), indent=1)

turn 1: {'prompt': 9781, 'cached': 0, 'out': 13} | The rover drove 11 m on Sol 12.
turn 2: {'prompt': 9806, 'cached': 0, 'out': 15} | The battery level was 42% on Sol 150.
turn 3: {'prompt': 9832, 'cached': 0, 'out': 32} | Sol 213, Sol 235, and Sol 251 had the most photos 
turn 4: {'prompt': 9872, 'cached': 0, 'out': 19} | A day on Mars (a sol) lasts 24 hours and 39 minute


turn 5: {'prompt': 9902, 'cached': 0, 'out': 15} | The dust opacity was 1.33 on Sol 77.


If you skipped the model cells, load the recorded usage (saved by the cell above on a real run). Then
compare hit rates and bills, with "uncached" pricing every prompt token at the full rate.

In [20]:
if "real_buggy" not in globals():          # recorded output of the two cells above
    saved = json.load(open("../data/recorded/41.json"))
    real_fixed, real_buggy = saved["fixed"], saved["buggy"]
for name, rows in [("fixed", real_fixed), ("buggy", real_buggy)]:
    rate = sum(r["cached"] for r in rows) / sum(r["prompt"] for r in rows)
    bill = sum(gemini_cost(**r) for r in rows)
    full = sum(gemini_cost(r["prompt"], 0, r["out"]) for r in rows)
    print(f"{name}: hit rate {rate:.0%} | cached per turn {[r['cached'] for r in rows]} | ${bill:.4f} vs ${full:.4f} uncached")

fixed: hit rate 82% | cached per turn [8160, 8152, 8144, 8136, 8128] | $0.0041 vs $0.0151 uncached
buggy: hit rate 0% | cached per turn [0, 0, 0, 0, 0] | $0.0150 vs $0.0150 uncached


Recorded run: the fixed chat read about 8.1k of its ~9.8k prompt tokens from the cache on every turn
(82%), and cost $0.0041 instead of $0.0151. Even turn 1 hit, because the same prefix had been sent a
few minutes earlier while recording; in an earlier recording turns 1 and 2 missed and turn 3 hit, so
implicit hits are best effort. The cached count stayed flat as the chat grew: Gemini reused the stable
start, not the new history. The buggy chat: **0 cached tokens on every turn**, same answers.

### 9. Debugging a 0% hit rate
Check, in order: is the prefix above the minimum, is it identical between calls, are the calls within
the TTL? The calls were seconds apart, so check the first two on the buggy run's requests.

In [21]:
def as_text(system: str, contents: list) -> list[dict]:
    return [{"text": json.dumps(SCHEMAS)}, {"text": system}] + [
        {"text": c.role + ": " + "".join(p.text or "" for p in c.parts)} for c in contents]

print("prompt above 4,096 tokens:", real_buggy[0]["prompt"] > 4096)
if "buggy_requests" in globals():          # needs the model cells (the requests are not saved)
    (s1, c1), (s2, c2) = buggy_requests[0], buggy_requests[1]
    print("buggy turn 1 vs 2:", first_difference(as_text(s1, c1), as_text(s2, c2)))
    (s1, c1), (s2, c2) = fixed_requests[0], fixed_requests[1]
    print("fixed turn 1 vs 2:", first_difference(as_text(s1, c1), as_text(s2, c2)))

prompt above 4,096 tokens: True
buggy turn 1 vs 2: identical for 166 tokens, then '00.\nYou are Scout, a rese' vs '30.\nYou are Scout, a rese'
fixed turn 1 vs 2: identical for 7893 tokens, then '' vs 'model: The rover drove 11'


Big enough, and seconds apart, so the culprit is the prefix. The buggy requests split after 166
tokens (the tool list, then the date), at the clock's seconds. The fixed ones are identical for the
whole of turn 1's request; turn 2 only adds text at the end. If all three checks pass and the hit
rate is still 0%, check the provider: implicit caching is best effort, and some setups do not offer
it.

### 10. Breakpoints on a real Anthropic request
For Anthropic you mark breakpoints yourself with `cache_control`. No Anthropic key was available while
building, so this is the request shape only. The order is tools, system, messages, as in the simulator.

In [22]:
anthropic_request = {
    "model": CACHE_RULES["model"], "max_tokens": 200,
    "tools": [{"name": s["name"], "description": s["description"], "input_schema": s["parameters"]}
              for s in SCHEMAS],
    "system": [{"type": "text", "text": SYSTEM_LONG, "cache_control": {"type": "ephemeral"}}],  # 5 min
    "messages": [{"role": "user", "content": [{"type": "text", "text": f"[sent 10:41:00] {QUESTIONS[0]}",
                                               "cache_control": {"type": "ephemeral"}}]}],
}
print("breakpoints:", json.dumps(anthropic_request).count('"cache_control"'),
      "| system block starts:", anthropic_request["system"][0]["text"][:40])

breakpoints: 2 | system block starts: You are Scout, a research assistant. Ans


Its usage would report `cache_creation_input_tokens`, `cache_read_input_tokens` and `input_tokens`
(tokens after the last breakpoint); total input is their sum, exactly the three columns the simulator
printed. `{"type": "ephemeral", "ttl": "1h"}` asks for the 1-hour TTL.

## At work
- **Rate limits count only uncached input on Anthropic,** so caching raises throughput too: 2M input
  tokens per minute at an 80% hit rate is about 10M total. Teams size quotas with the hit rate →
  lesson 54.
- **Hit rate is a logged metric,** next to tokens and cost, per call and per feature; a sudden drop
  usually means someone edited the top of a prompt or a tool description → lesson 49.
- **Changing tools, `tool_choice` or thinking settings mid-chat invalidates the cache**, so production
  agents fix them per conversation; clearing old tool results also breaks the prefix → lesson 32.
- **Caches are isolated per organisation** (per workspace on the Claude API), and Vertex caching can be
  turned off per project for zero data retention → lesson 50.

## What just happened
- A toy KV cache showed why only a **prefix** can be reused: one early change made almost nothing reusable.
- `PrefixCache` followed Anthropic's rules: breakpoints, lookback, 512-token minimum, 5-minute TTL,
  writes at 1.25x and reads at 0.05x.
- With a stable prefix, Scout's 5-turn chat hit about 80% and the bill fell by about three quarters.
- A timestamp in the system prompt gave 0% and cost more than no caching; `first_difference` found
  it; moving the time into the user message fixed it.
- Long pauses (TTL) and short prompts (minimum size) also miss, silently.
- On real Gemini, `cached_content_token_count` showed the same: 82% with the timestamp at the end,
  0% with it at the top, identical answers. You only see it if you measure.

## Common mistakes
**1. Doing arithmetic on `cached_content_token_count` directly.** With no hit it is `None`, not 0.

In [23]:
u = types.GenerateContentResponseUsageMetadata(prompt_token_count=9971)      # a turn with no cache hit
uncached = u.prompt_token_count - u.cached_content_token_count

TypeError: unsupported operand type(s) for -: 'int' and 'NoneType'

Fix: default to 0, as step 8 does: `u.cached_content_token_count or 0`.

**2. A breakpoint on every block.** It feels safe, but the API allows 4 (Anthropic returns 400); the
simulator raises the same way.

In [24]:
history = [{"role": "user" if i % 2 == 0 else "assistant", "text": f"message {i}"} for i in range(5)]
PrefixCache().call([{**b, "cache": "5m"} for b in build_request(history)], now=0)

ValueError: 7 cache breakpoints; the API allows 4

Fix: one breakpoint after the stable part and one moving breakpoint on the newest message. More are
only useful for parts that change at different speeds (tools rarely, notes daily, chat every turn).

In [25]:
print(PrefixCache().call(build_request(history), now=0))

{'input': 0, 'cache_write': 7890, 'cache_read': 0}


## Try it
1. A teammate wants Scout to greet the user by name, so the system prompt now starts with
   `"User: Asha.\n"`. Run `sim_chat` with that system prompt. Does the hit rate drop? Why not?
2. Now the name changes every turn (a shared kiosk). Measure the hit rate, then fix it without
   losing the name, and print the hit rate and total cost of the fixed version.
3. Using `real_buggy`, compute what the buggy Gemini chat would have cost with the fixed run's
   cached token counts, and how much the timestamp bug would cost per month at 5,000 such chats a day.

In [26]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [27]:
llm_replay.stop()

## Key terms
- **prompt caching** — the provider reuses the work done on the start of a recent request when a new request begins with exactly the same tokens, and bills those tokens at a fraction of the input price.
- **cache prefix** — the part of a request, from the first token up to a breakpoint, that is stored and matched exactly; order is tools, system, then messages.
- **cache breakpoint** — a marker (Anthropic `cache_control`) saying "cache the request up to here"; at most 4 per request.
- **cache TTL** — how long a cache entry lives without use: 5 minutes by default on Anthropic (refreshed on each hit), 1 hour for a dearer write.
- **cache hit rate** — cached input tokens divided by all input tokens, measured from each response's usage.
- **KV cache (outside view)** — the per-token internal state a model computes for a prompt; each token's state depends on all earlier tokens, so only an identical prefix can be reused.